In [0]:
bronze_df = spark.read.table("mashebhebs_nyctaxi.bronze_nyctaxi")

In [0]:
import pyspark.sql.functions as F

possible_data = (
    (F.col("tpep_pickup_datetime") < F.col("tpep_dropoff_datetime")) &
    (F.col("trip_distance") > 0.0) &
    (F.col("fare_amount") > 0.0) &
    (F.col("pickup_zip") > 0) &
    (F.col("dropoff_zip") > 0) & 
    (F.col("ingestion_time").isNotNull()) &
    (F.col("tpep_pickup_datetime").isNotNull()) &
    (F.col("tpep_dropoff_datetime").isNotNull()) &
    (F.col("trip_distance").isNotNull()) &
    (F.col("fare_amount").isNotNull()) &
    (F.col("pickup_zip").isNotNull()) &
    (F.col("dropoff_zip").isNotNull())
)

cleaned_df = bronze_df.filter(possible_data)

In [0]:
time_diff_minutes = F.timestamp_diff("MINUTE", F.col("tpep_pickup_datetime"), F.col("tpep_dropoff_datetime")).cast("long")

silver_df = cleaned_df.withColumn("trip_duration",
    F.concat(
        F.floor(time_diff_minutes/60).cast("string"), 
        F.lit("h"),
        (time_diff_minutes%60).cast("string")                   
    )
)

display(silver_df.limit(10))

In [0]:
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("mashebhebs_nyctaxi.silver_nyctaxi")
